In [6]:
LOAN_AMOUNT = 400_000
INTEREST_RATE = 3.45
MONTHLY_PAYMENT = 2_000.00

ANNUAL_EXTRA_PAYMENT = 3_000
ANNUAL_EXTRA_PAYMENT_MONTH = 6 # June
ANNUAL_EXTRA_PAYMENT_COUNT = -1 # -1 = until loan is paid off

In [7]:
import pandas as pd
from datetime import datetime, timedelta

# Read the existing data to determine the starting point
df_existing = pd.read_csv('data.csv')
last_row = df_existing.iloc[-1]

# Parse the starting date
start_date = pd.to_datetime(last_row['Date'])
remaining_balance = LOAN_AMOUNT - last_row['Cumulative Principal Paid']
total_interest_paid = last_row['Cumulative Interest Paid']
total_principal_paid = last_row['Cumulative Principal Paid']
total_paid = last_row['Cumulative Total Paid']

print(f"Starting from {start_date.strftime('%Y-%m')}")
print(f"Remaining balance: ${remaining_balance:,.2f}")

Starting from 2026-07
Remaining balance: $396,586.83


In [8]:
# List to store all calculations
calculations = []

# Add all existing payments from data.csv
for idx, row in df_existing.iterrows():
    # Parse Principal Paid % - handle both decimal and percentage formats
    principal_paid_pct = row['Principal Paid %']
    if isinstance(principal_paid_pct, str) and '%' in principal_paid_pct:
        # Already formatted as percentage
        principal_paid_pct_str = principal_paid_pct.strip()
    else:
        # Decimal format - convert to percentage
        percentage_value = float(principal_paid_pct) * 100
        # Format with up to 3 decimal places, removing trailing zeros
        principal_paid_pct_str = f"{percentage_value:.3g} %"

    calculations.append({
        'Date': row['Date'],
        'Type': row['Type'],
        'Payment Amount': row['Payment Amount'],
        'Interest': row['Interest'],
        'Principal': row['Principal'],
        'Principal Remaining': round(float(row['Principal Remaining']), 2),
        'Cumulative Interest Paid': row['Cumulative Interest Paid'],
        'Cumulative Principal Paid': row['Cumulative Principal Paid'],
        'Cumulative Total Paid': row['Cumulative Total Paid'],
        'Principal Paid %': principal_paid_pct_str
    })

# Calculate remaining months
monthly_rate = INTEREST_RATE / 100 / 12
current_date = start_date + timedelta(days=32)  # Move to next month
current_date = current_date.replace(day=1)  # Ensure first day of month

month_count = 0
extra_payment_count = 0  # Track number of extra payments made

while remaining_balance > 0.01:  # Account for rounding errors
    month_count += 1

    # Calculate interest for this month
    interest_payment = remaining_balance * monthly_rate

    # Determine principal payment for regular monthly payment
    regular_principal_payment = MONTHLY_PAYMENT - interest_payment

    # Check if loan will be paid off this month
    if regular_principal_payment >= remaining_balance:
        regular_principal_payment = remaining_balance
        regular_paid_amount = interest_payment + regular_principal_payment
    else:
        regular_paid_amount = MONTHLY_PAYMENT

    # Update balances for monthly payment
    remaining_balance -= regular_principal_payment
    total_principal_paid += regular_principal_payment
    total_interest_paid += interest_payment
    total_paid = total_interest_paid + total_principal_paid

    # Calculate percentage of principal paid
    paid_principal_percentage = total_principal_paid / LOAN_AMOUNT

    # Format date as YYYY-MM
    date_str = current_date.strftime('%Y-%m')

    # Add monthly payment to calculations
    calculations.append({
        'Date': date_str,
        'Type': 'Monthly',
        'Payment Amount': round(regular_paid_amount, 2),
        'Interest': round(interest_payment, 2),
        'Principal': round(regular_principal_payment, 2),
        'Principal Remaining': round(remaining_balance, 2),
        'Cumulative Interest Paid': round(total_interest_paid, 2),
        'Cumulative Principal Paid': round(total_principal_paid, 2),
        'Cumulative Total Paid': round(total_paid, 2),
        'Principal Paid %': f"{paid_principal_percentage * 100:.3g} %"
    })

    # Check if it's the annual extra payment month and if loan isn't paid off yet
    # Honor ANNUAL_EXTRA_PAYMENT_COUNT: -1 means unlimited, otherwise limit to that count
    can_make_extra_payment = (ANNUAL_EXTRA_PAYMENT_COUNT == -1 or extra_payment_count < ANNUAL_EXTRA_PAYMENT_COUNT)

    if current_date.month == ANNUAL_EXTRA_PAYMENT_MONTH and remaining_balance > 0.01 and ANNUAL_EXTRA_PAYMENT > 0 and can_make_extra_payment:
        extra_principal_payment = min(ANNUAL_EXTRA_PAYMENT, remaining_balance)

        # Update balances for extra payment
        remaining_balance -= extra_principal_payment
        total_principal_paid += extra_principal_payment
        total_paid = total_interest_paid + total_principal_paid

        # Increment extra payment counter
        extra_payment_count += 1

        # Calculate updated percentage
        paid_principal_percentage = total_principal_paid / LOAN_AMOUNT

        # Add extra payment to calculations
        calculations.append({
            'Date': date_str,
            'Type': 'Additional',
            'Payment Amount': round(extra_principal_payment, 2),
            'Interest': 0.0,
            'Principal': round(extra_principal_payment, 2),
            'Principal Remaining': round(remaining_balance, 2),
            'Cumulative Interest Paid': round(total_interest_paid, 2),
            'Cumulative Principal Paid': round(total_principal_paid, 2),
            'Cumulative Total Paid': round(total_paid, 2),
            'Principal Paid %': f"{paid_principal_percentage * 100:.3g} %"
        })

    # Move to next month
    if current_date.month == 12:
        current_date = current_date.replace(year=current_date.year + 1, month=1)
    else:
        current_date = current_date.replace(month=current_date.month + 1)

    # Safety check to prevent infinite loops
    if month_count > 600:  # 50 years max
        print("Warning: Reached maximum month limit")
        break

# Write to CSV
df_calculations = pd.DataFrame(calculations)
df_calculations.to_csv('results/calculation.csv', index=False)

print(f"✓ Mortgage calculation complete!")
print(f"Total months: {len(calculations)}")
print(f"Total interest paid: ${total_interest_paid:,.2f}")
print(f"Total amount paid: ${total_paid:,.2f}")
print(f"\nCalculations saved to 'calculation.csv'")

✓ Mortgage calculation complete!
Total months: 272
Total interest paid: $163,046.95
Total amount paid: $563,046.95

Calculations saved to 'calculation.csv'


In [9]:
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_RIGHT
from datetime import datetime, timedelta

# Create PDF with landscape orientation for better table fit
pdf_filename = 'results/calculation.pdf'
doc = SimpleDocTemplate(pdf_filename, pagesize=landscape(letter), topMargin=0.5*inch, bottomMargin=0.5*inch)

# Container for PDF elements
elements = []

# Add title
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'CustomTitle',
    parent=styles['Heading1'],
    fontSize=16,
    textColor=colors.HexColor('#1f4788'),
    spaceAfter=12,
    alignment=TA_CENTER
)
title = Paragraph("Mortgage Calculation Report", title_style)
elements.append(title)

# Add summary info with Euro currency
summary_style = ParagraphStyle(
    'Summary',
    parent=styles['Normal'],
    fontSize=9,
    spaceAfter=8
)
summary_text = f"""
<b>Loan Amount:</b> € {LOAN_AMOUNT:,.2f} |
<b>Interest Rate:</b> {INTEREST_RATE}% |
<b>Monthly Payment:</b> € {MONTHLY_PAYMENT:,.2f} |
<b>Annual Extra Payment:</b> € {ANNUAL_EXTRA_PAYMENT:,.2f} (in {datetime(2026, ANNUAL_EXTRA_PAYMENT_MONTH, 1).strftime('%B')})
<br/>
<b>Total Months:</b> {len(df_calculations)} |
<b>Total Interest Paid:</b> € {total_interest_paid:,.2f} |
<b>Total Amount Paid:</b> € {total_paid:,.2f}
"""
elements.append(Paragraph(summary_text, summary_style))
elements.append(Spacer(1, 0.15*inch))

# Calculate scenario without extra payments
monthly_rate = INTEREST_RATE / 100 / 12
remaining_without_extra = LOAN_AMOUNT - last_row['Cumulative Principal Paid']
interest_without_extra = remaining_without_extra * monthly_rate
total_months_without_extra = 0
total_interest_without_extra = last_row['Cumulative Interest Paid']

# Simulate without extra payments
temp_balance = remaining_without_extra
while temp_balance > 0.01 and total_months_without_extra < 500:
    total_months_without_extra += 1
    interest_payment = temp_balance * monthly_rate
    principal_payment = MONTHLY_PAYMENT - interest_payment

    if principal_payment >= temp_balance:
        principal_payment = temp_balance

    temp_balance -= principal_payment
    total_interest_without_extra += interest_payment

# Calculate difference
current_months = len(df_calculations[df_calculations['Type'] == 'Monthly'])  # Monthly payment count
months_saved = total_months_without_extra - current_months
years_saved = months_saved // 12
months_remainder = months_saved % 12
interest_saved = total_interest_without_extra - total_interest_paid

# Add comparison section
comparison_style = ParagraphStyle(
    'Comparison',
    parent=styles['Normal'],
    fontSize=10,
    spaceAfter=10,
    textColor=colors.HexColor('#2d5016')
)
comparison_box = f"""
<b>Effect of Annual Extra Payments (€ {ANNUAL_EXTRA_PAYMENT:,.2f} in {datetime(2026, ANNUAL_EXTRA_PAYMENT_MONTH, 1).strftime('%B')}):</b><br/>
✓ <b>Loan paid off {years_saved} years and {months_remainder} months earlier</b> ({months_saved} months saved)<br/>
✓ <b>Interest saved: € {interest_saved:,.2f}</b>
"""
elements.append(Paragraph(comparison_box, comparison_style))
elements.append(Spacer(1, 0.2*inch))

# Prepare table data
table_data = [df_calculations.columns.tolist()]  # Header row
num_existing_rows = len(df_existing)  # Number of rows from data.csv file

# Add data rows with Euro currency
for idx, row in df_calculations.iterrows():
    table_data.append([
        str(row['Date']),
        str(row['Type']),
        f"€ {float(row['Payment Amount']):,.2f}",
        f"€ {float(row['Interest']):,.2f}",
        f"€ {float(row['Principal']):,.2f}",
        f"€ {float(row['Principal Remaining']):,.2f}",
        f"€ {float(row['Cumulative Interest Paid']):,.2f}",
        f"€ {float(row['Cumulative Principal Paid']):,.2f}",
        f"€ {float(row['Cumulative Total Paid']):,.2f}",
        str(row['Principal Paid %'])
    ])

# Create table with styling
table = Table(table_data, colWidths=[0.7*inch, 0.7*inch, 1*inch, 0.9*inch, 0.9*inch, 1*inch, 1.1*inch, 1.1*inch, 1.1*inch, 0.8*inch])

# Apply table styling
table_style = TableStyle([
    # Header styling
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1f4788')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 7),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 8),

    # Row styling
    ('ALIGN', (0, 1), (-1, -1), 'RIGHT'),
    ('ALIGN', (1, 1), (1, -1), 'CENTER'),
    ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
    ('FONTSIZE', (0, 1), (-1, -1), 7),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f0f0f0')]),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ('TOPPADDING', (0, 1), (-1, -1), 4),
    ('BOTTOMPADDING', (0, 1), (-1, -1), 4),
])

# Highlight rows from data.csv in light green (paid records)
for row_idx in range(1, num_existing_rows + 1):
    table_style.add('BACKGROUND', (0, row_idx), (-1, row_idx), colors.HexColor('#e8f5e9'))

# Highlight "Additional" payment rows in light orange
for idx, row in df_calculations.iterrows():
    if row['Type'] == 'Additional':
        table_style.add('BACKGROUND', (0, idx + 1), (-1, idx + 1), colors.HexColor('#fff4e6'))

table.setStyle(table_style)
elements.append(table)

# Build PDF
doc.build(elements)

print(f"✓ PDF report created: {pdf_filename}")
print(f"  - Loan paid off {years_saved} years and {months_remainder} months earlier")
print(f"  - Interest saved: € {interest_saved:,.2f}")


✓ PDF report created: results/calculation.pdf
  - Loan paid off 3 years and 7 months earlier
  - Interest saved: € 33,057.98


In [10]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from datetime import datetime, timedelta

# Prepare data for "With Extra Payments" scenario
with_extra_dates = []
with_extra_balances = []

for idx, row in df_calculations.iterrows():
    if row['Type'] == 'Monthly':  # Only include monthly payments, not additional ones
        try:
            date = pd.to_datetime(row['Date'])
            with_extra_dates.append(date)
            with_extra_balances.append(float(row['Principal Remaining']))
        except:
            pass

# Prepare data for "Without Extra Payments" scenario
without_extra_dates = []
without_extra_balances = [LOAN_AMOUNT - last_row['Cumulative Principal Paid']]
without_extra_dates.append(start_date)

temp_balance = LOAN_AMOUNT - last_row['Cumulative Principal Paid']
current_date_temp = start_date + timedelta(days=32)
current_date_temp = current_date_temp.replace(day=1)
month_idx = 0

while temp_balance > 0.01 and month_idx < 500:
    month_idx += 1
    interest_payment = temp_balance * monthly_rate
    principal_payment = MONTHLY_PAYMENT - interest_payment

    if principal_payment >= temp_balance:
        principal_payment = temp_balance

    temp_balance -= principal_payment
    without_extra_balances.append(max(0, temp_balance))
    without_extra_dates.append(current_date_temp)

    if current_date_temp.month == 12:
        current_date_temp = current_date_temp.replace(year=current_date_temp.year + 1, month=1)
    else:
        current_date_temp = current_date_temp.replace(month=current_date_temp.month + 1)

# Create the burndown chart
fig, ax = plt.subplots(figsize=(12, 7))

# Plot both scenarios
ax.plot(with_extra_dates, with_extra_balances, linewidth=2.5, label='With Annual Extra Payments (€5,000)',
        color='#2d5016', marker='o', markersize=3, markevery=12)  # Mark every 12 months (yearly)
ax.plot(without_extra_dates, without_extra_balances, linewidth=2.5, label='Without Extra Payments',
        color='#d32f2f', linestyle='--', marker='s', markersize=3, markevery=12)

# Formatting
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Principal Remaining (€)', fontsize=12, fontweight='bold')
ax.set_title('Mortgage Burndown Chart: Impact of Annual Extra Payments', fontsize=14, fontweight='bold', pad=20)

# Format y-axis with Euro currency
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x/1000:.0f}K'))

# Format x-axis with dates
ax.xaxis.set_major_locator(mdates.YearLocator(2))  # Every 2 years
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45, ha='right')

# Add grid
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax.set_axisbelow(True)

# Add legend
ax.legend(loc='upper right', fontsize=11, framealpha=0.95)

# Add annotation for time saved
mid_date = with_extra_dates[len(with_extra_dates)//2] if with_extra_dates else without_extra_dates[len(without_extra_dates)//2]
ax.text(0.5, 0.05, f'Time Saved: {years_saved} years {months_remainder} months | Interest Saved: €{interest_saved:,.0f}',
        transform=ax.transAxes, fontsize=11, ha='center', va='bottom',
        bbox=dict(boxstyle='round', facecolor='#e8f5e9', alpha=0.8, edgecolor='#2d5016', linewidth=1.5))

# Tight layout
plt.tight_layout()

# Save to PDF
pdf_filename_burndown = 'results/burndown.pdf'
with PdfPages(pdf_filename_burndown) as pdf:
    pdf.savefig(fig, bbox_inches='tight')
    d = pdf.infodict()
    d['Title'] = 'Mortgage Burndown Chart'
    d['Author'] = 'Mortgage Calculator'
    d['Subject'] = 'Comparison of mortgage payoff with and without extra payments'
    d['CreationDate'] = datetime.now()

plt.close()

print(f"✓ Burndown chart created: {pdf_filename_burndown}")
print(f"  - Chart compares scenarios with and without €{ANNUAL_EXTRA_PAYMENT:,.0f} annual extra payments")
print(f"  - Time saved: {years_saved} years {months_remainder} months")


✓ Burndown chart created: results/burndown.pdf
  - Chart compares scenarios with and without €3,000 annual extra payments
  - Time saved: 3 years 7 months
